<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/RenameCalvingVideos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rename the calving videos and their output folders

One naming rule for every calving video **and** its folder in `CalvingPenPipelineOutputs`:

**`CalvingPen<pen><Mon><day>Cow<n>[ROI]`**: pen, month and day (no year, no leading zero), the recording number in brackets as the cow number (1 if none), `ROI` for cropped clips.

| Now | New name |
|---|---|
| `2026-11-11 00-00-00 Calving Pen 3 (2).avi` | `CalvingPen3Nov11Cow2.avi` |
| `2026-11-11 00-00-00 Calving Pen 3.avi` | `CalvingPen3Nov11Cow1.avi` |
| `2026-10-29 09-48-06 Calving Pen 2 (3).avi` | `CalvingPen2Oct29Cow3.avi` |
| `2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.avi` | `CalvingPen3Sep26Cow1ROI.avi` |
| output folder `2026-09-26 00-00-00 Calving Pen 2` | `CalvingPen2Sep26Cow1` |
| output folder `CalvingPen1Nov11Part1Win2` (automatic window) | `CalvingPen1Nov11Cow1Win2` |

**What is renamed:** video files in `CalvingPenVideos ` and `CalvingPenPipeline` (and their subfolders), and the video folders in `CalvingPenPipelineOutputs` (`Pool`, `PoolTest`, `Old` stay). Inside a renamed output folder the stored paths and names are updated too, so finished steps stay finished. Notebooks and spreadsheets keep their names.

**Safe:** cell 3 first **only shows** the plan (two tables: videos, output folders); nothing changes until `Apply = True`. Left alone: `Check` (a `Clip...` file or no date/pen in the name: rename by hand), `Unreadable` (broken shortcut or still uploading), `Conflict` (two files would get the same name, e.g. two ROI clips of one video, or the name is taken). Every rename goes into `CalvingPenPipelineOutputs/RenameLog.csv`; `Undo = True` puts the old names back. The pipeline finds a video by its old name (through the log) and by its new name before renaming.

**One person runs this, once, while nobody is running the pipeline** (a file renamed while a step reads it would fail). No GPU needed. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Rename import PlanRenames, PlanFolderRenames, ApplyRenames, UndoRenames, RenameLog
from CalvingPipeline.Paths import GroupDir, OutputRoot
import pandas as pd
pd.set_option("display.max_rows", 500, "display.max_colwidth", 120)
print("Group folder:", GroupDir, "exists" if GroupDir.exists() else "NOT FOUND")

In [ ]:
# 2. Settings: the only cell to edit.
Folders = ["CalvingPenVideos ", "CalvingPenPipeline"]   # video folders inside ANSC4040GroupProject (note the space after Videos)
RenameOutputFolders = True   # also rename the video folders in CalvingPenPipelineOutputs
Apply = False                # False = only show the plan; True = rename everything marked "Rename"
Undo = False                 # True = put the old names back (from RenameLog.csv) instead

In [ ]:
# 3. Run: show the plan; rename only with Apply = True (or undo with Undo = True).
if Undo:
    UndoRenames(RenameLog)
else:
    Plans = {"Videos": PlanRenames(Folders)}
    if RenameOutputFolders:
        Plans["Output folders"] = PlanFolderRenames(OutputRoot)
    for Title, Plan in Plans.items():
        print(f"\n{Title}:", Plan.Status.value_counts().to_dict())
        display(Plan)
    if Apply:
        for Title, Plan in Plans.items():
            print(f"\n{Title}:")
            ApplyRenames(Plan, RenameLog)
    else:
        print("\nPREVIEW ONLY: nothing renamed. Check the NewName columns, then set Apply = True and rerun cells 2-3.")

## About this notebook

**Rule** (`CalvingName` in `CalvingPen/CalvingPipeline/Rename.py`, from the group's naming notebook): `CalvingPen<pen><Mon><day>Cow<n>[ROI]`. The number in brackets after the pen (`Calving Pen 3 (2)`) is the cow number; without one it is `Cow1`. A start time, ROI time window or fps in the old name is dropped (only `ROI` is kept), so two ROI clips of one video get the same name and are left as `Conflict`. An automatic-window folder keeps its `Win1`, `Win2` ending.

**Status column:** `Rename` (will be renamed), `Same` (already right), `Check` (left alone: rename by hand), `Unreadable` (broken Drive shortcut or still uploading), `Conflict` (left alone).

**Afterwards:** use the new names in `PipelineSteps` (`VideoFile`); old names still find the video. Cow labels in the pool are `Pen2Cow1`, `Pen2Cow2`, ... per pen in date order (`CowRegister.csv`); a video's `Cow<n>` tells the pool which cow of that pen and day its tracks are.